In [0]:
# Install Folium for map visualization
%pip install folium --quiet
print("✓ Folium installed successfully!")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
✓ Folium installed successfully!


In [0]:
# CELL 1 — Imports
import folium
from folium import plugins
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, max as spark_max
from datetime import datetime, timedelta

print("Imports loaded successfully!")

Imports loaded successfully!


In [0]:
# CELL 2A — Create Pincode Coordinates Delta Table (Run Once)

# Static coordinate data for 50 pincodes (25 state capitals × 2)
pincode_data = [
    # Amaravati, Andhra Pradesh
    (522503, 16.5062, 80.6480, 'Amaravati', 'Guntur', 'Andhra Pradesh'),
    (522020, 16.5735, 80.3567, 'Amaravati', 'Guntur', 'Andhra Pradesh'),
    # Itanagar, Arunachal Pradesh
    (791111, 27.1020, 93.6920, 'Itanagar', 'Papum Pare', 'Arunachal Pradesh'),
    (791113, 27.0844, 93.6053, 'Itanagar', 'Papum Pare', 'Arunachal Pradesh'),
    # Dispur/Guwahati, Assam
    (781005, 26.1433, 91.7898, 'Guwahati', 'Kamrup Metropolitan', 'Assam'),
    (781006, 26.1445, 91.7362, 'Guwahati', 'Kamrup Metropolitan', 'Assam'),
    # Patna, Bihar
    (800001, 25.5941, 85.1376, 'Patna', 'Patna', 'Bihar'),
    (800004, 25.6093, 85.1417, 'Patna', 'Patna', 'Bihar'),
    # Raipur, Chhattisgarh
    (492001, 21.2514, 81.6296, 'Raipur', 'Raipur', 'Chhattisgarh'),
    (492007, 21.2379, 81.6337, 'Raipur', 'Raipur', 'Chhattisgarh'),
    # Panaji, Goa
    (403001, 15.4909, 73.8278, 'Panaji', 'North Goa', 'Goa'),
    (403002, 15.4989, 73.8246, 'Panaji', 'North Goa', 'Goa'),
    # Gandhinagar, Gujarat
    (382010, 23.2156, 72.6369, 'Gandhinagar', 'Gandhinagar', 'Gujarat'),
    (382011, 23.2232, 72.6425, 'Gandhinagar', 'Gandhinagar', 'Gujarat'),
    # Chandigarh
    (160001, 30.7333, 76.7794, 'Chandigarh', 'Chandigarh', 'Chandigarh'),
    (160017, 30.7500, 76.7900, 'Chandigarh', 'Chandigarh', 'Chandigarh'),
    # Shimla, Himachal Pradesh
    (171001, 31.1048, 77.1734, 'Shimla', 'Shimla', 'Himachal Pradesh'),
    (171002, 31.0856, 77.1711, 'Shimla', 'Shimla', 'Himachal Pradesh'),
    # Ranchi, Jharkhand
    (834001, 23.3441, 85.3096, 'Ranchi', 'Ranchi', 'Jharkhand'),
    (834002, 23.3701, 85.3376, 'Ranchi', 'Ranchi', 'Jharkhand'),
    # Bengaluru, Karnataka
    (560001, 12.9716, 77.5946, 'Bengaluru', 'Bangalore', 'Karnataka'),
    (560002, 12.9698, 77.6025, 'Bengaluru', 'Bangalore', 'Karnataka'),
    # Thiruvananthapuram, Kerala
    (695001, 8.5241, 76.9366, 'Thiruvananthapuram', 'Thiruvananthapuram', 'Kerala'),
    (695014, 8.4875, 76.9525, 'Thiruvananthapuram', 'Thiruvananthapuram', 'Kerala'),
    # Bhopal, Madhya Pradesh
    (462001, 23.2599, 77.4126, 'Bhopal', 'Bhopal', 'Madhya Pradesh'),
    (462003, 23.2443, 77.4116, 'Bhopal', 'Bhopal', 'Madhya Pradesh'),
    # Mumbai, Maharashtra
    (400001, 18.9388, 72.8354, 'Mumbai', 'Mumbai City', 'Maharashtra'),
    (400002, 18.9570, 72.8258, 'Mumbai', 'Mumbai City', 'Maharashtra'),
    # Imphal, Manipur
    (795001, 24.8170, 93.9368, 'Imphal', 'Imphal West', 'Manipur'),
    (795004, 24.8066, 93.9449, 'Imphal', 'Imphal West', 'Manipur'),
    # Shillong, Meghalaya
    (793001, 25.5788, 91.8933, 'Shillong', 'East Khasi Hills', 'Meghalaya'),
    (793002, 25.5744, 91.8789, 'Shillong', 'East Khasi Hills', 'Meghalaya'),
    # Aizawl, Mizoram
    (796001, 23.7271, 92.7176, 'Aizawl', 'Aizawl', 'Mizoram'),
    (796007, 23.7461, 92.7175, 'Aizawl', 'Aizawl', 'Mizoram'),
    # Kohima, Nagaland
    (797001, 25.6586, 94.1086, 'Kohima', 'Kohima', 'Nagaland'),
    (797003, 25.6751, 94.1169, 'Kohima', 'Kohima', 'Nagaland'),
    # Bhubaneswar, Odisha
    (751001, 20.2961, 85.8245, 'Bhubaneswar', 'Khordha', 'Odisha'),
    (751002, 20.2700, 85.8370, 'Bhubaneswar', 'Khordha', 'Odisha'),
    # Jaipur, Rajasthan
    (302001, 26.9124, 75.7873, 'Jaipur', 'Jaipur', 'Rajasthan'),
    (302003, 26.8983, 75.8095, 'Jaipur', 'Jaipur', 'Rajasthan'),
    # Chennai, Tamil Nadu
    (600001, 13.0827, 80.2707, 'Chennai', 'Chennai', 'Tamil Nadu'),
    (600003, 13.0732, 80.2609, 'Chennai', 'Chennai', 'Tamil Nadu'),
    # Hyderabad, Telangana
    (500001, 17.3850, 78.4867, 'Hyderabad', 'Hyderabad', 'Telangana'),
    (500002, 17.3987, 78.4738, 'Hyderabad', 'Hyderabad', 'Telangana'),
    # Agartala, Tripura
    (799001, 23.8315, 91.2868, 'Agartala', 'West Tripura', 'Tripura'),
    (799006, 23.8403, 91.2665, 'Agartala', 'West Tripura', 'Tripura'),
    # Lucknow, Uttar Pradesh
    (226001, 26.8467, 80.9462, 'Lucknow', 'Lucknow', 'Uttar Pradesh'),
    (226002, 26.8590, 80.9290, 'Lucknow', 'Lucknow', 'Uttar Pradesh'),
    # Kolkata, West Bengal
    (700001, 22.5726, 88.3639, 'Kolkata', 'Kolkata', 'West Bengal'),
    (700007, 22.5448, 88.3426, 'Kolkata', 'Kolkata', 'West Bengal'),
]

# Create DataFrame with schema
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType

schema = StructType([
    StructField("pincode", IntegerType(), False),
    StructField("latitude", DoubleType(), False),
    StructField("longitude", DoubleType(), False),
    StructField("city", StringType(), False),
    StructField("district", StringType(), False),
    StructField("state", StringType(), False)
])

pincode_coords_df = spark.createDataFrame(pincode_data, schema)

# Save as Delta table in workspace.epialert schema (where you have permissions)
table_name = "workspace.epialert.epi_pincode_coordinates"
pincode_coords_df.write.format("delta").mode("overwrite").saveAsTable(table_name)

print(f"✓ Created Delta table: {table_name}")
print(f"  • Total pincodes: {pincode_coords_df.count()}")
print(f"  • Covers 25 state capitals × 2 pincodes each")
print(f"\n✓ Table can now be used across notebooks and dashboards!")

# Display sample
print("\nSample coordinates:")
display(spark.table(table_name).limit(10))

✓ Created Delta table: workspace.epialert.epi_pincode_coordinates
  • Total pincodes: 50
  • Covers 25 state capitals × 2 pincodes each

✓ Table can now be used across notebooks and dashboards!

Sample coordinates:


pincode,latitude,longitude,city,district,state
522503,16.5062,80.648,Amaravati,Guntur,Andhra Pradesh
522020,16.5735,80.3567,Amaravati,Guntur,Andhra Pradesh
791111,27.102,93.692,Itanagar,Papum Pare,Arunachal Pradesh
791113,27.0844,93.6053,Itanagar,Papum Pare,Arunachal Pradesh
781005,26.1433,91.7898,Guwahati,Kamrup Metropolitan,Assam
781006,26.1445,91.7362,Guwahati,Kamrup Metropolitan,Assam
800001,25.5941,85.1376,Patna,Patna,Bihar
800004,25.6093,85.1417,Patna,Patna,Bihar
492001,21.2514,81.6296,Raipur,Raipur,Chhattisgarh
492007,21.2379,81.6337,Raipur,Raipur,Chhattisgarh


In [0]:
# CELL 3 — Load Pincode Coordinates from Delta Table
import requests
import time

# Load pincode coordinates from Delta table
table_name = "workspace.epialert.epi_pincode_coordinates"

try:
    pincode_table_df = spark.table(table_name)
    pincode_dict = {str(row.pincode): row.asDict() for row in pincode_table_df.collect()}
    print(f"✓ Loaded {len(pincode_dict)} pincodes from Delta table: {table_name}")
except Exception as e:
    print(f"⚠ Could not load table {table_name}: {e}")
    print("⚠ Please run Cell 2A first to create the table")
    pincode_dict = {}

# Cache for lookups
pincode_cache = {}

def get_pincode_coordinates(pincode):
    """
    Fetch latitude, longitude, and location details for a pincode.
    Strategy: 1) Check cache, 2) Try Delta table, 3) Fall back to API
    """
    pincode_str = str(pincode)
    
    # Check cache first
    if pincode_str in pincode_cache:
        return pincode_cache[pincode_str]
    
    # Try Delta table
    if pincode_str in pincode_dict:
        row = pincode_dict[pincode_str]
        result = {
            'pincode': row['pincode'],
            'latitude': row['latitude'],
            'longitude': row['longitude'],
            'area_name': row['city'],
            'district': row['district'],
            'state': row['state'],
            'city': row['city']
        }
        pincode_cache[pincode_str] = result
        return result
    
    # Try API as fallback (for pincodes not in table)
    try:
        url = f"https://api.postalpincode.in/pincode/{pincode_str}"
        response = requests.get(url, timeout=3)
        
        if response.status_code == 200:
            data = response.json()
            
            if data and len(data) > 0 and data[0].get('Status') == 'Success':
                post_offices = data[0].get('PostOffice', [])
                
                if post_offices and len(post_offices) > 0:
                    office = post_offices[0]
                    
                    result = {
                        'pincode': pincode,
                        'latitude': float(office.get('Latitude', 0)) if office.get('Latitude') else None,
                        'longitude': float(office.get('Longitude', 0)) if office.get('Longitude') else None,
                        'area_name': office.get('Name', 'Unknown'),
                        'district': office.get('District', 'Unknown'),
                        'state': office.get('State', 'Unknown'),
                        'city': office.get('District', 'Unknown')
                    }
                    
                    if result['latitude'] and result['longitude']:
                        pincode_cache[pincode_str] = result
                        return result
        
        return None
        
    except Exception as e:
        return None

# Get unique pincodes from anomaly data
unique_pincodes = spark.sql("""
    SELECT DISTINCT pincode 
    FROM workspace.default.epi_alert_gold_anomaly
    WHERE alert_flag = 1
    ORDER BY pincode
""").toPandas()['pincode'].tolist()

print(f"\nFound {len(unique_pincodes)} unique pincodes with alerts")
print(f"Fetching coordinates (Delta table + API fallback)...\n")

# Fetch coordinates for all pincodes
pincode_lookup_list = []
table_count = 0
api_count = 0

for i, pincode in enumerate(unique_pincodes):
    coords = get_pincode_coordinates(pincode)
    if coords and coords['latitude'] and coords['longitude']:
        pincode_lookup_list.append(coords)
        source = "💾 TABLE" if str(pincode) in pincode_dict else "🌐 API"
        if str(pincode) in pincode_dict:
            table_count += 1
        else:
            api_count += 1
        print(f"✓ {i+1}/{len(unique_pincodes)}: {pincode} → {coords['city']}, {coords['state']} ({coords['latitude']:.4f}, {coords['longitude']:.4f}) {source}")
    else:
        print(f"✗ {i+1}/{len(unique_pincodes)}: {pincode} → Coordinates not found")
    
    # Small delay only for API calls
    if i < len(unique_pincodes) - 1 and str(pincode) not in pincode_dict:
        time.sleep(0.1)

# Convert to Spark DataFrame for downstream processing
if pincode_lookup_list:
    pincode_lookup_df = spark.createDataFrame(pincode_lookup_list)
    print(f"\n✓ Successfully fetched {len(pincode_lookup_list)}/{len(unique_pincodes)} pincode coordinates")
    print(f"  • From Delta table: {table_count}")
    print(f"  • From API: {api_count}")
    display(pincode_lookup_df.limit(10))
else:
    print("\n✗ No coordinates found.")
    pincode_lookup_df = None

✓ Loaded 50 pincodes from Delta table: workspace.epialert.epi_pincode_coordinates

Found 50 unique pincodes with alerts
Fetching coordinates (Delta table + API fallback)...

✓ 1/50: 160001 → Chandigarh, Chandigarh (30.7333, 76.7794) 💾 TABLE
✓ 2/50: 160017 → Chandigarh, Chandigarh (30.7500, 76.7900) 💾 TABLE
✓ 3/50: 171001 → Shimla, Himachal Pradesh (31.1048, 77.1734) 💾 TABLE
✓ 4/50: 171002 → Shimla, Himachal Pradesh (31.0856, 77.1711) 💾 TABLE
✓ 5/50: 226001 → Lucknow, Uttar Pradesh (26.8467, 80.9462) 💾 TABLE
✓ 6/50: 226002 → Lucknow, Uttar Pradesh (26.8590, 80.9290) 💾 TABLE
✓ 7/50: 302001 → Jaipur, Rajasthan (26.9124, 75.7873) 💾 TABLE
✓ 8/50: 302003 → Jaipur, Rajasthan (26.8983, 75.8095) 💾 TABLE
✓ 9/50: 382010 → Gandhinagar, Gujarat (23.2156, 72.6369) 💾 TABLE
✓ 10/50: 382011 → Gandhinagar, Gujarat (23.2232, 72.6425) 💾 TABLE
✓ 11/50: 400001 → Mumbai, Maharashtra (18.9388, 72.8354) 💾 TABLE
✓ 12/50: 400002 → Mumbai, Maharashtra (18.9570, 72.8258) 💾 TABLE
✓ 13/50: 403001 → Panaji, Goa (15.4

area_name,city,district,latitude,longitude,pincode,state
Chandigarh,Chandigarh,Chandigarh,30.7333,76.7794,160001,Chandigarh
Chandigarh,Chandigarh,Chandigarh,30.75,76.79,160017,Chandigarh
Shimla,Shimla,Shimla,31.1048,77.1734,171001,Himachal Pradesh
Shimla,Shimla,Shimla,31.0856,77.1711,171002,Himachal Pradesh
Lucknow,Lucknow,Lucknow,26.8467,80.9462,226001,Uttar Pradesh
Lucknow,Lucknow,Lucknow,26.859,80.929,226002,Uttar Pradesh
Jaipur,Jaipur,Jaipur,26.9124,75.7873,302001,Rajasthan
Jaipur,Jaipur,Jaipur,26.8983,75.8095,302003,Rajasthan
Gandhinagar,Gandhinagar,Gandhinagar,23.2156,72.6369,382010,Gujarat
Gandhinagar,Gandhinagar,Gandhinagar,23.2232,72.6425,382011,Gujarat


In [0]:
# CELL 3 — Load Alert Data from Gold Table (WITH DISEASE PREDICTIONS)

# Get latest alerts per pincode (ALL available data for demo)
alerts_df = spark.sql("""
    WITH latest_alerts AS (
        SELECT 
            pincode,
            symptom_cluster,
            primary_disease,
            primary_disease_prob,
            report_date,
            daily_count,
            `30d_avg` as baseline,
            count_vs_30d_avg,
            if_score,
            alert_flag,
            risk_level,
            ROW_NUMBER() OVER (
                PARTITION BY pincode, symptom_cluster 
                ORDER BY report_date DESC
            ) as rn
        FROM workspace.default.epi_alert_gold_anomaly
    )
    SELECT 
        pincode,
        symptom_cluster,
        primary_disease,
        ROUND(primary_disease_prob, 3) as disease_confidence,
        report_date,
        daily_count,
        ROUND(baseline, 1) as baseline,
        ROUND(count_vs_30d_avg, 2) as spike_ratio,
        ROUND(if_score, 3) as anomaly_score,
        alert_flag,
        risk_level
    FROM latest_alerts
    WHERE rn = 1 AND alert_flag = 1
""")

print(f"✓ Loaded {alerts_df.count()} active alerts (latest per pincode-symptom)")
print("\nTop alerts by disease confidence:")
display(alerts_df.orderBy(col("disease_confidence").desc(), col("spike_ratio").desc()).limit(10))

✓ Loaded 9 active alerts (latest per pincode-symptom)

Top alerts by disease confidence:


pincode,symptom_cluster,primary_disease,disease_confidence,report_date,daily_count,baseline,spike_ratio,anomaly_score,alert_flag,risk_level
462001,cough_cold,null,null,2024-12-26T00:00:00.000Z,3,1.2,2.57,-0.614,1,high
695001,cough_cold,null,null,2024-12-28T00:00:00.000Z,3,1.3,2.31,-0.634,1,high
500001,cough_only,null,null,2024-12-28T00:00:00.000Z,3,1.3,2.25,-0.636,1,high
522020,cough_only,null,null,2024-12-31T00:00:00.000Z,3,1.4,2.19,-0.601,1,high
797003,cough_only,null,null,2024-12-31T00:00:00.000Z,3,1.4,2.14,-0.602,1,high
781005,cough_only,null,null,2024-12-29T00:00:00.000Z,3,1.4,2.09,-0.625,1,high
700001,other,null,null,2024-12-31T00:00:00.000Z,3,1.9,1.61,-0.6,1,high
171001,other,null,null,2024-12-31T00:00:00.000Z,3,2.1,1.45,-0.602,1,high
171002,other,null,null,2024-12-31T00:00:00.000Z,2,2.1,0.94,-0.602,1,high


In [0]:
# CELL 4 — Join Alerts with Pincode Locations (DISEASE-BASED AGGREGATION)

# Create temp view from API-fetched coordinates
if pincode_lookup_df is not None:
    pincode_lookup_df.createOrReplaceTempView("pincode_locations_temp")
    
    # Join and aggregate by pincode, focusing on diseases
    map_data = spark.sql("""
        WITH pincode_alerts AS (
            SELECT 
                a.pincode,
                l.latitude,
                l.longitude,
                l.city,
                l.state,
                COUNT(*) as alert_count,
                MAX(CASE WHEN a.risk_level = 'high' THEN 1 ELSE 0 END) as has_high_risk,
                COLLECT_LIST(CONCAT(a.primary_disease, ' (', ROUND(a.disease_confidence * 100, 0), '%)')) as diseases_with_confidence,
                COLLECT_LIST(a.primary_disease) as diseases,
                COLLECT_LIST(a.daily_count) as daily_counts,
                MAX(a.spike_ratio) as max_spike_ratio,
                MIN(a.anomaly_score) as min_anomaly_score
            FROM (
                SELECT pincode, symptom_cluster, primary_disease, primary_disease_prob as disease_confidence,
                       daily_count, spike_ratio, anomaly_score, risk_level
                FROM (
                    SELECT 
                        pincode,
                        symptom_cluster,
                        primary_disease,
                        primary_disease_prob,
                        report_date,
                        daily_count,
                        `30d_avg` as baseline,
                        count_vs_30d_avg as spike_ratio,
                        if_score as anomaly_score,
                        alert_flag,
                        risk_level,
                        ROW_NUMBER() OVER (
                            PARTITION BY pincode, symptom_cluster 
                            ORDER BY report_date DESC
                        ) as rn
                    FROM workspace.default.epi_alert_gold_anomaly
                )
                WHERE rn = 1 AND alert_flag = 1
            ) a
            JOIN pincode_locations_temp l
            ON a.pincode = l.pincode
            GROUP BY a.pincode, l.latitude, l.longitude, l.city, l.state
        )
        SELECT 
            pincode,
            latitude,
            longitude,
            city,
            state,
            alert_count,
            CASE 
                WHEN has_high_risk = 1 THEN 'high'
                ELSE 'medium'
            END as overall_risk,
            diseases_with_confidence,
            diseases,
            daily_counts,
            max_spike_ratio,
            min_anomaly_score
        FROM pincode_alerts
    """)
    
    # Convert to pandas for Folium
    map_df = map_data.toPandas()
    
    print(f"✓ Prepared {len(map_df)} locations for mapping")
    print(f"  • High risk: {(map_df['overall_risk'] == 'high').sum()}")
    print(f"  • Medium risk: {(map_df['overall_risk'] == 'medium').sum()}")
    print("\nTop 5 locations by spike ratio:")
    if len(map_df) > 0:
        display(map_df.nlargest(5, 'max_spike_ratio')[['city', 'pincode', 'alert_count', 'overall_risk', 'max_spike_ratio']])
    else:
        print("No alert locations found.")
else:
    print("✗ Cannot create map: No pincode coordinates available")
    map_df = pd.DataFrame()  # Empty dataframe

✓ Prepared 9 locations for mapping
  • High risk: 9
  • Medium risk: 0

Top 5 locations by spike ratio:


city,pincode,alert_count,overall_risk,max_spike_ratio
Bhopal,462001,1,high,2.569226377390808
Thiruvananthapuram,695001,1,high,2.3059185242121445
Hyderabad,500001,1,high,2.248313764676493
Amaravati,522020,1,high,2.19351693882525
Kohima,797003,1,high,2.1413276231263385


In [0]:
# CELL 5 — Create Enhanced Folium Map with Heatmap (DISEASE-FOCUSED VISUALIZATION)

# Calculate map center (India center)
if len(map_df) > 0:
    center_lat = map_df['latitude'].mean()
    center_lon = map_df['longitude'].mean()
else:
    center_lat, center_lon = 23.0, 79.0  # India center

# Create base map with better styling
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=5,
    tiles='CartoDB positron',  # Clean, professional look
    prefer_canvas=True
)

# Enhanced color mapping with gradient effects
risk_colors = {
    'high': '#DC143C',      # Crimson red
    'medium': '#FF8C00',    # Dark orange
    'low': '#FFD700',       # Gold
    'normal': '#32CD32'     # Lime green
}

# Disease emoji mapping
disease_emojis = {
    'dengue': '🦟', 'chikungunya': '🦟', 'malaria': '🦟',
    'cholera': '🤢', 'typhoid': '🤒', 'gastroenteritis': '🤢',
    'influenza': '🤧', 'common_cold': '🤧', 'pneumonia': '🫁',
    'covid_like': '😷', 'measles': '🤒', 'leptospirosis': '🤒',
    'food_poisoning': '🤢'
}

# ═══════════════════════════════════════════════════════════════
# 1. ADD HEATMAP LAYER (for regional density visualization)
# ═══════════════════════════════════════════════════════════════
if len(map_df) > 0:
    # Prepare heatmap data: [lat, lon, weight]
    heat_data = []
    for idx, row in map_df.iterrows():
        # Weight by spike ratio and alert count for intensity
        weight = row['max_spike_ratio'] * row['alert_count']
        heat_data.append([row['latitude'], row['longitude'], weight])
    
    # Add heatmap layer
    from folium.plugins import HeatMap
    HeatMap(
        heat_data,
        min_opacity=0.3,
        max_zoom=13,
        radius=25,
        blur=30,
        gradient={
            0.0: '#00FF00',   # Green (low)
            0.3: '#FFFF00',   # Yellow
            0.5: '#FFA500',   # Orange
            0.7: '#FF4500',   # Orange-red
            1.0: '#DC143C'    # Crimson (high)
        }
    ).add_to(m)

# ═══════════════════════════════════════════════════════════════
# 2. ADD MARKER CLUSTER FOR BETTER ORGANIZATION
# ═══════════════════════════════════════════════════════════════
from folium.plugins import MarkerCluster

# Create marker cluster with custom styling
marker_cluster = MarkerCluster(
    name='Alert Markers',
    overlay=True,
    control=True,
    icon_create_function="""
    function(cluster) {
        var childCount = cluster.getChildCount();
        var c = ' marker-cluster-';
        if (childCount < 3) {
            c += 'small';
        } else if (childCount < 6) {
            c += 'medium';
        } else {
            c += 'large';
        }
        return new L.DivIcon({ 
            html: '<div><span>' + childCount + '</span></div>', 
            className: 'marker-cluster' + c, 
            iconSize: new L.Point(40, 40) 
        });
    }
    """
).add_to(m)

# ═══════════════════════════════════════════════════════════════
# 3. ADD ENHANCED MARKERS WITH BETTER STYLING
# ═══════════════════════════════════════════════════════════════
for idx, row in map_df.iterrows():
    # Format disease list with emojis
    diseases_formatted = []
    for disease_with_conf in row['diseases_with_confidence'][:5]:  # Top 5
        disease_name = disease_with_conf.split(' (')[0]
        emoji = disease_emojis.get(disease_name, '🦠')
        diseases_formatted.append(f"{emoji} {disease_with_conf}")
    
    diseases_html = '<br>'.join(diseases_formatted)
    if len(row['diseases_with_confidence']) > 5:
        diseases_html += f"<br><i>+ {len(row['diseases_with_confidence']) - 5} more</i>"
    
    total_cases = sum(row['daily_counts'])
    
    # Enhanced popup HTML with better styling
    popup_html = f"""
    <div style="font-family: 'Segoe UI', Arial, sans-serif; width: 300px;">
        <div style="background: linear-gradient(135deg, {risk_colors[row['overall_risk']]}22, {risk_colors[row['overall_risk']]}44); 
                    padding: 12px; border-radius: 8px 8px 0 0; margin: -10px -10px 10px -10px;">
            <h3 style="margin: 0; color: #333; font-size: 18px;">🚨 {row['city']}, {row['state']}</h3>
        </div>
        <div style="padding: 0 5px;">
            <table style="width: 100%; font-size: 13px; border-collapse: collapse;">
                <tr><td style="padding: 4px 0; color: #666;"><b>Pincode:</b></td><td style="text-align: right;">{row['pincode']}</td></tr>
                <tr><td style="padding: 4px 0; color: #666;"><b>Risk Level:</b></td>
                    <td style="text-align: right;">
                        <span style="background: {risk_colors[row['overall_risk']]}; color: white; 
                                     padding: 2px 8px; border-radius: 12px; font-weight: bold; font-size: 11px;">
                            {row['overall_risk'].upper()}
                        </span>
                    </td></tr>
                <tr><td style="padding: 4px 0; color: #666;"><b>Active Alerts:</b></td><td style="text-align: right; font-weight: bold;">{row['alert_count']}</td></tr>
                <tr><td style="padding: 4px 0; color: #666;"><b>Total Cases:</b></td><td style="text-align: right; font-weight: bold; color: #DC143C;">{total_cases}</td></tr>
                <tr><td style="padding: 4px 0; color: #666;"><b>Max Spike:</b></td><td style="text-align: right;">{row['max_spike_ratio']:.2f}x baseline</td></tr>
                <tr><td style="padding: 4px 0; color: #666;"><b>Anomaly Score:</b></td><td style="text-align: right;">{row['min_anomaly_score']:.3f}</td></tr>
            </table>
            <div style="margin-top: 12px; padding-top: 8px; border-top: 1px solid #eee;">
                <p style="margin: 0 0 6px 0; font-weight: bold; color: #333; font-size: 13px;">🦠 Predicted Diseases:</p>
                <div style="margin-left: 8px; font-size: 12px; line-height: 1.6;">{diseases_html}</div>
            </div>
        </div>
    </div>
    """
    
    # Calculate marker size based on severity
    base_size = 12
    size_multiplier = 1 + (row['max_spike_ratio'] / 5)  # Scale with spike ratio
    marker_radius = min(base_size + (row['alert_count'] * 2 * size_multiplier), 30)  # Cap at 30
    
    # Add enhanced circle marker
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=marker_radius,
        popup=folium.Popup(popup_html, max_width=340),
        tooltip=f"{row['city']} - {row['alert_count']} alert(s) - Risk: {row['overall_risk'].upper()}",
        color='white',  # White border for contrast
        fill=True,
        fillColor=risk_colors[row['overall_risk']],
        fillOpacity=0.8,
        weight=2.5,
        className='pulse-marker'  # For animation effect
    ).add_to(marker_cluster)

# ═══════════════════════════════════════════════════════════════
# 4. ADD ENHANCED LEGEND
# ═══════════════════════════════════════════════════════════════
legend_html = '''
<div style="position: fixed; 
            top: 10px; right: 10px; width: 220px;
            background: linear-gradient(135deg, #ffffff 0%, #f8f9fa 100%);
            box-shadow: 0 4px 12px rgba(0,0,0,0.15);
            z-index: 9999; 
            border: none;
            border-radius: 12px;
            padding: 16px; 
            font-family: 'Segoe UI', Arial, sans-serif;">
    <h4 style="margin: 0 0 12px 0; color: #333; font-size: 16px; border-bottom: 2px solid #DC143C; padding-bottom: 8px;">
        🦠 Disease Outbreak Map
    </h4>
    <div style="margin-bottom: 12px;">
        <p style="margin: 0 0 6px 0; font-size: 11px; color: #666; font-weight: bold;">RISK LEVELS:</p>
        <div style="display: flex; align-items: center; margin: 4px 0;">
            <span style="display: inline-block; width: 16px; height: 16px; border-radius: 50%; 
                         background: #DC143C; margin-right: 8px;"></span>
            <span style="font-size: 13px; color: #333;">High Risk</span>
        </div>
        <div style="display: flex; align-items: center; margin: 4px 0;">
            <span style="display: inline-block; width: 16px; height: 16px; border-radius: 50%; 
                         background: #FF8C00; margin-right: 8px;"></span>
            <span style="font-size: 13px; color: #333;">Medium Risk</span>
        </div>
        <div style="display: flex; align-items: center; margin: 4px 0;">
            <span style="display: inline-block; width: 16px; height: 16px; border-radius: 50%; 
                         background: #FFD700; margin-right: 8px;"></span>
            <span style="font-size: 13px; color: #333;">Low Risk</span>
        </div>
    </div>
    <div style="border-top: 1px solid #ddd; padding-top: 8px; margin-top: 8px;">
        <p style="margin: 0 0 4px 0; font-size: 11px; color: #666; font-weight: bold;">MARKER SIZE:</p>
        <p style="margin: 0; font-size: 11px; color: #666; line-height: 1.5;">
            Larger markers = higher alert count and severity
        </p>
    </div>
    <div style="border-top: 1px solid #ddd; padding-top: 8px; margin-top: 8px;">
        <p style="margin: 0 0 4px 0; font-size: 11px; color: #666; font-weight: bold;">HEATMAP:</p>
        <p style="margin: 0; font-size: 11px; color: #666; line-height: 1.5;">
            Red zones = high outbreak intensity
        </p>
    </div>
</div>
'''

m.get_root().html.add_child(folium.Element(legend_html))

# Add custom CSS for pulsing animation
css = """
<style>
@keyframes pulse {
    0% { transform: scale(1); opacity: 0.8; }
    50% { transform: scale(1.1); opacity: 1; }
    100% { transform: scale(1); opacity: 0.8; }
}
.pulse-marker {
    animation: pulse 2s ease-in-out infinite;
}
.pulse-marker-new {
    animation: pulse 1s ease-in-out infinite;
}
</style>
"""
m.get_root().html.add_child(folium.Element(css))

print(f"✓ Map created with {len(map_df)} locations marked")
print(f"✓ Risk distribution: {map_df['overall_risk'].value_counts().to_dict()}")
print(f"✓ Total active alerts: {map_df['alert_count'].sum()}")

# Define map_obj for use in subsequent cells
map_obj = m

✓ Map created with 9 locations marked
✓ Risk distribution: {'high': 9}
✓ Total active alerts: 9


In [0]:
# CELL 6 — Display the Map

displayHTML(map_obj._repr_html_())

Make this Notebook Trusted to load map: File -> Trust Notebook <iframe srcdoc="<!DOCTYPE html>
<html>
<head>
 
 <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
 <script src="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.js"></script>
 <script src="https://code.jquery.com/jquery-3.7.1.min.js"></script>
 <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.2.2/dist/js/bootstrap.bundle.min.js"></script>
 <script src="https://cdnjs.cloudflare.com/ajax/libs/Leaflet.awesome-markers/2.0.2/leaflet.awesome-markers.js"></script>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@5.2.2/dist/css/bootstrap.min.css"/>
 <link rel="stylesheet" href="https://netdna.bootstrapcdn.com/bootstrap/3.0.0/css/bootstrap-glyphicons.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/@fortawesome/fontawesome-free@6.2.0/css/all.min.css"/>
 <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/Leaflet.awesome-markers/2.0.2/leaflet.awesome-markers.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/gh/python-visualization/folium/folium/templates/leaflet.awesome.rotate.min.css"/>
 
 <meta name="viewport" content="width=device-width,
 initial-scale=1.0, maximum-scale=1.0, user-scalable=no" />
 <style>
 #map_15174150cdc54800c7b3882cfa053fd3 {
 position: relative;
 width: 100.0%;
 height: 100.0%;
 left: 0.0%;
 top: 0.0%;
 }
 .leaflet-container { font-size: 1rem; }
 </style>

 <style>html, body {
 width: 100%;
 height: 100%;
 margin: 0;
 padding: 0;
 }
 </style>

 <style>#map {
 position:absolute;
 top:0;
 bottom:0;
 right:0;
 left:0;
 }
 </style>

 <script>
 L_NO_TOUCH = false;
 L_DISABLE_3D = false;
 </script>

 
 <script src="https://cdn.jsdelivr.net/gh/python-visualization/folium@main/folium/templates/leaflet_heat.min.js"></script>
 <script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet.markercluster/1.1.0/leaflet.markercluster.js"></script>
 <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet.markercluster/1.1.0/MarkerCluster.css"/>
 <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet.markercluster/1.1.0/MarkerCluster.Default.css"/>
</head>
<body>
 
 
<div style="position: fixed; 
 top: 10px; right: 10px; width: 220px;
 background: linear-gradient(135deg, #ffffff 0%, #f8f9fa 100%);
 box-shadow: 0 4px 12px rgba(0,0,0,0.15);
 z-index: 9999; 
 border: none;
 border-radius: 12px;
 padding: 16px; 
 font-family: 'Segoe UI', Arial, sans-serif;">
 <h4 style="margin: 0 0 12px 0; color: #333; font-size: 16px; border-bottom: 2px solid #DC143C; padding-bottom: 8px;">
 🦠 Disease Outbreak Map
 </h4>
 <div style="margin-bottom: 12px;">
 <p style="margin: 0 0 6px 0; font-size: 11px; color: #666; font-weight: bold;">RISK LEVELS:</p>
 <div style="display: flex; align-items: center; margin: 4px 0;">
 <span style="display: inline-block; width: 16px; height: 16px; border-radius: 50%; 
 background: #DC143C; margin-right: 8px;"></span>
 <span style="font-size: 13px; color: #333;">High Risk</span>
 </div>
 <div style="display: flex; align-items: center; margin: 4px 0;">
 <span style="display: inline-block; width: 16px; height: 16px; border-radius: 50%; 
 background: #FF8C00; margin-right: 8px;"></span>
 <span style="font-size: 13px; color: #333;">Medium Risk</span>
 </div>
 <div style="display: flex; align-items: center; margin: 4px 0;">
 <span style="display: inline-block; width: 16px; height: 16px; border-radius: 50%; 
 background: #FFD700; margin-right: 8px;"></span>
 <span style="font-size: 13px; color: #333;">Low Risk</span>
 </div>
 </div>
 <div style="border-top: 1px solid #ddd; padding-top: 8px; margin-top: 8px;">
 <p style="margin: 0 0 4px 0; font-size: 11px; color: #666; font-weight: bold;">MARKER SIZE:</p>
 <p style="margin: 0; font-size: 11px; color: #666; line-height: 1.5;">
 Larger markers = higher alert count and severity
 </p>
 </div>

In [0]:
def plot_pincode_on_map(pincode, map_obj=None):
    """
    Plots a given pincode on the map with location details.
    Args:
        pincode (int or str): Pincode to plot.
        map_obj (folium.Map, optional): Folium map object to add marker to. If None, creates a new map centered at pincode.
    Returns:
        folium.Map: Map object with marker added.
    """
    coords = get_pincode_coordinates(pincode)
    if coords and coords['latitude'] and coords['longitude']:
        city = coords['city']
        state = coords['state']
        lat = coords['latitude']
        lon = coords['longitude']
        if map_obj is None:
            map_obj = folium.Map(
                location=[lat, lon],
                zoom_start=12,
                tiles='CartoDB positron'
            )
        popup_html = f"""
        <div style="font-family: 'Segoe UI', Arial, sans-serif; width: 220px;">
            <h4 style="margin: 0 0 8px 0; color: #333;">📍 {city}, {state}</h4>
            <table style="width: 100%; font-size: 13px;">
                <tr><td><b>Pincode:</b></td><td style="text-align: right;">{pincode}</td></tr>
                <tr><td><b>Latitude:</b></td><td style="text-align: right;">{lat:.4f}</td></tr>
                <tr><td><b>Longitude:</b></td><td style="text-align: right;">{lon:.4f}</td></tr>
            </table>
        </div>
        """
        folium.CircleMarker(
            location=[lat, lon],
            radius=12,
            popup=folium.Popup(popup_html, max_width=240),
            tooltip=f"{city}, {state} (Pincode: {pincode})",
            color='blue',
            fill=True,
            fillColor='blue',
            fillOpacity=0.7,
            weight=2
        ).add_to(map_obj)
        return map_obj
    else:
        print(f"✗ Coordinates not found for pincode {pincode}")
        return None

In [0]:
# Pipeline: Input data → anomaly detection → output pincode(s) with anomaly

def detect_anomaly_pincode(input_df, model_weights):
    """
    Detects anomalies in input data using model weights and returns pincodes with anomalies.
    Args:
        input_df (pyspark.sql.DataFrame): Input data with columns ['pincode', 'symptom_cluster', 'daily_count', 'baseline', ...]
        model_weights (dict): Weights for anomaly detection (e.g., {'spike_ratio': 0.6, 'anomaly_score': 0.4})
    Returns:
        List of pincodes where anomaly is detected.
    """
    # Calculate spike_ratio and anomaly_score using weights
    df = input_df.withColumn(
        "spike_ratio", col("daily_count") / col("baseline")
    ).withColumn(
        "anomaly_score", 
        model_weights.get('spike_ratio', 0.5) * col("spike_ratio") +
        model_weights.get('other', 0.5) * lit(0)  # Add other features if needed
    )
    
    # Define anomaly threshold (customize as needed)
    anomaly_condition = (col("spike_ratio") > 2.0) & (col("anomaly_score") > 0.8)
    anomalies_df = df.filter(anomaly_condition)
    
    # Output pincodes with anomaly
    anomaly_pincodes = [row.pincode for row in anomalies_df.select("pincode").distinct().collect()]
    return anomaly_pincodes

# Example usage:
# input_df = spark.createDataFrame([...])  # Your input data
# model_weights = {'spike_ratio': 0.7, 'other': 0.3}
# anomaly_pincodes = detect_anomaly_pincode(input_df, model_weights)
# print("Anomaly detected at pincodes:", anomaly_pincodes)


In [0]:
# CELL 7 — Save Disease Outbreak Map for Streamlit Dashboard

# Save to file
map_path = "/tmp/epi_alert_map.html"
map_obj.save(map_path)

print(f"✓ Map saved to: {map_path}")
print("\n📦 Ready for Streamlit dashboard integration!")
print("\n🦠 Disease-Focused Map Statistics:")
print(f"  • Total alert locations: {len(map_df)}")
print(f"  • High risk zones: {(map_df['overall_risk'] == 'high').sum()}")
print(f"  • Medium risk zones: {(map_df['overall_risk'] == 'medium').sum()}")
print(f"  • Total active disease alerts: {map_df['alert_count'].sum()}")
print(f"  • Max spike ratio: {map_df['max_spike_ratio'].max():.2f}x baseline")

# Show top diseases detected
if len(map_df) > 0:
    all_diseases = []
    for diseases_list in map_df['diseases']:
        all_diseases.extend(diseases_list)
    
    from collections import Counter
    disease_counts = Counter(all_diseases)
    print("\n💉 Top Detected Diseases:")
    for disease, count in disease_counts.most_common(5):
        print(f"  • {disease.replace('_', ' ').title()}: {count} alert(s)")

✓ Map saved to: /tmp/epi_alert_map.html

📦 Ready for Streamlit dashboard integration!

🦠 Disease-Focused Map Statistics:
  • Total alert locations: 9
  • High risk zones: 9
  • Medium risk zones: 0
  • Total active disease alerts: 9
  • Max spike ratio: 2.57x baseline

💉 Top Detected Diseases:
